# 02 — Text Embeddings

Install and load `sentence-transformers`, generate embeddings for all 4,681 products, and save to disk.

In [1]:
# Cell 1 — Install
%pip install --break-system-packages sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Cell 2 — Load model
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model loaded successfully")
print(f"Max sequence length: {model.max_seq_length}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully
Max sequence length: 256


In [3]:
# Cell 3 — Load data
import pandas as pd

df = pd.read_csv("../data/processed/products_final.csv")
print(f"Loaded {len(df)} products")
print("Columns:", df.columns.tolist())

Loaded 4681 products
Columns: ['uniq_id', 'crawl_timestamp', 'product_url', 'product_name', 'product_category_tree', 'pid', 'retail_price', 'discounted_price', 'image', 'is_FK_Advantage_product', 'description', 'product_rating', 'overall_rating', 'brand', 'product_specifications', 'main_category', 'image_url', 'image_path']


In [4]:
# Cell 4 — Build combined text field
def build_text(row):
    parts = []
    for col in ["product_name", "brand", "description"]:
        val = row.get(col, "")
        if pd.notna(val) and str(val).strip():
            parts.append(str(val).strip())
    return " | ".join(parts)

df["text_for_embedding"] = df.apply(build_text, axis=1)

print(f"Sample text: {df['text_for_embedding'].iloc[0][:120]}")
print(f"Total texts: {len(df)}")

Sample text: Sukuma Women's Leggings | Unknown | Sukuma Women's Leggings - Buy Purple, Black Sukuma Women's Leggings For Only Rs. 999
Total texts: 4681


In [5]:
# Cell 5 — Generate embeddings for ALL products
all_texts = df["text_for_embedding"].tolist()

embeddings = model.encode(
    all_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embeddings generated!")
print("Shape:", embeddings.shape)

Batches:   0%|          | 0/147 [00:00<?, ?it/s]

Embeddings generated!
Shape: (4681, 384)


In [6]:
# Cell 6 — Save embeddings to disk
import os

save_path = "../data/processed/text_embeddings.npy"
np.save(save_path, embeddings)

size_mb = os.path.getsize(save_path) / (1024 * 1024)
print(f"Text embeddings saved successfully!")
print(f"Path:  {save_path}")
print(f"Shape: {embeddings.shape}")
print(f"Size:  {size_mb:.1f} MB")

Text embeddings saved successfully!
Path:  ../data/processed/text_embeddings.npy
Shape: (4681, 384)
Size:  6.9 MB


In [7]:
# Cell 7 — Sanity check: semantic search over all products
from sentence_transformers.util import cos_sim

query = "women's cycling shorts"
query_embedding = model.encode(query, convert_to_numpy=True, normalize_embeddings=True)

scores = cos_sim(query_embedding, embeddings)[0].numpy()
top_idx = scores.argsort()[::-1][:5]

print(f"Top 5 results for: '{query}'\n")
for rank, idx in enumerate(top_idx, 1):
    print(f"{rank}. [{scores[idx]:.4f}] {all_texts[idx][:100]}")

Top 5 results for: 'women's cycling shorts'

1. [0.6445] Alisha Solid Women's Cycling Shorts | Alisha | Alisha Solid Women's Cycling Shorts - Buy Black, Whit
2. [0.5643] RIPR Self Design Women's Multicolor Basic Shorts | RIPR | Key Features of RIPR Self Design Women's M
3. [0.5374] Sportking Women's Leggings | Unknown | Sportking Women's Leggings - Buy Yellow Sportking Women's Leg
4. [0.5290] Mynte Solid Women's Cycling Shorts, Gym Shorts, Swim Shorts | Mynte | Specifications of Mynte Solid 
5. [0.5146] Mynte Solid Women's Cycling Shorts, Gym Shorts, Swim Shorts | Mynte | Key Features of Mynte Solid Wo
